In [1]:

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ast

In [2]:
from google.colab import files
uploaded = files.upload()


Saving tmdb_5000_movies.csv.zip to tmdb_5000_movies.csv.zip
Saving tmdb_5000_credits.csv.zip to tmdb_5000_credits.csv.zip


In [3]:
!unzip tmdb_5000_movies.csv.zip

Archive:  tmdb_5000_movies.csv.zip
  inflating: tmdb_5000_movies.csv    


In [4]:
!unzip tmdb_5000_credits.csv.zip

Archive:  tmdb_5000_credits.csv.zip
  inflating: tmdb_5000_credits.csv   


In [5]:
movies = pd.read_csv('/content/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/tmdb_5000_credits.csv')

print("Movies loaded:", len(movies))
print("Credits loaded:", len(credits))

Movies loaded: 4803
Credits loaded: 4803


In [10]:
credits.columns = ['id', 'title', 'cast', 'crew']
movies = movies.merge(credits, on='title')

print("Total movies after merge:", len(movies))

Total movies after merge: 4818


In [8]:
!ls


sample_data	       tmdb_5000_credits.csv.zip  tmdb_5000_movies.csv.zip
tmdb_5000_credits.csv  tmdb_5000_movies.csv


In [12]:
print(movies.columns.tolist())

['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast_x', 'crew_x', 'id', 'cast_y', 'crew_y']


In [13]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast_x', 'crew_x']].copy()
movies.rename(columns={'cast_x': 'cast', 'crew_x': 'crew'}, inplace=True)
movies.dropna(inplace=True)

print(movies.shape)

(4818, 7)


In [14]:
import ast

def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]

def convert_cast(text):
    return [i['name'] for i in ast.literal_eval(text)[:3]]

def fetch_director(text):
    return [i['name'] for i in ast.literal_eval(text) if i['job'] == 'Director']

movies['genres']   = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast']     = movies['cast'].apply(convert_cast)
movies['crew']     = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

print("Done!")

Done!


In [15]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

movies['cast']     = movies['cast'].apply(collapse)
movies['crew']     = movies['crew'].apply(collapse)
movies['genres']   = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

new_df = movies[['movie_id', 'title', 'tags']]
new_df.head(3)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...


In [18]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

print(vectors.shape)

(4818, 5000)


In [19]:
similarity = cosine_similarity(vectors)

print(similarity.shape)

(4818, 4818)


In [20]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    print(f"\nBecause you liked '{movie}', you might also like:\n")
    for i, score in movies_list:
        print(f"  • {new_df.iloc[i].title}  (similarity: {round(score, 2)})")

In [26]:
recommend("Batman Begins")
recommend("Interstellar")


Because you liked 'Batman Begins', you might also like:

  • The Dark Knight  (similarity: 0.38)
  • The Dark Knight Rises  (similarity: 0.36)
  • Batman  (similarity: 0.33)
  • Batman  (similarity: 0.33)
  • Batman & Robin  (similarity: 0.31)

Because you liked 'Interstellar', you might also like:

  • Silent Running  (similarity: 0.23)
  • The Martian  (similarity: 0.22)
  • Space Pirate Captain Harlock  (similarity: 0.22)
  • Apollo 13  (similarity: 0.22)
  • Space Cowboys  (similarity: 0.21)


In [32]:
new_df[new_df['title'].str.contains("The Notebook", case=False)]

,movie_id,title,tags
1565,11036,The Notebook,an epic love story centered around an older ma...


In [33]:
recommend("The Notebook")


Because you liked 'The Notebook', you might also like:

  • Lovely, Still  (similarity: 0.31)
  • Veer-Zaara  (similarity: 0.31)
  • Mississippi Mermaid  (similarity: 0.3)
  • The Age of Adaline  (similarity: 0.29)
  • Next Stop Wonderland  (similarity: 0.29)
